<h1 style="color:DodgerBlue">Индивидуальный проект</h1>


<h2 style="color:DodgerBlue">Название проекта: База данных заказов</h2>

----

### Вариант задания 15


Требования к производным классам:
1. СтандартнаяСтрока (StandardLine): Должна содержать дополнительные
атрибуты, такие как Количество единиц (Units). Метод CalculateTotal() должен
быть переопределен для учета количества единиц при расчете общей
стоимости.
2. СпециальнаяСтрока (SpecialLine): Должна содержать дополнительные
атрибуты, такие как Скидка (Discount). Метод UpdatePrice() должен быть
переопределен для применения скидки к цене товара.
3. БесплатнаяСтрока (FreeLine): Должна содержать дополнительные
атрибуты, такие как Предварительный платеж (Prepayment). Метод CalculateTotal()
должен быть переопределен для учета предварительного платежа при расчете общей
стоимости.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

Что добавлено в реализации:
- простое наследование: классы StandardLine, SpecialLine и FreeLine наследуют
  напрямую от базового класса OrderLine;
- сложное наследование: класс FamilyPackLine наследует от StandardLine, тем самым
  выстраивается цепочка OrderLine -> StandardLine -> FamilyPackLine;
- множественное наследование: классы SpecialLine и FreeLine одновременно
  наследуют от OrderLine и реализуют интерфейс IBonus;
- 3-4 атрибута и метода в базовом и производных классах;
- взаимодействие объектов: класс Order принимает строки заказа через AddLine(),
  суммирует стоимость через CalculateTotal() и начисляет бонусы через интерфейс IBonus.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
// ============================================================
// Sprint2/Task3: Индивидуальный проект "База данных заказов"
// Вариант 15. Простое + сложное + множественное наследование
// ============================================================

// Создаём строки заказа
Order order = new Order();
order.AddLine(new StandardLine(1, "Молоко 3,2%", 89m, 3));
order.AddLine(new SpecialLine(2, "Сыр Маасдам", 290m, 15));
order.AddLine(new FreeLine(3, "Йогурт клубника", 65m, 30m));
order.AddLine(new FamilyPackLine(4, "Чай Ассам 100x30", 450m, 2, 10));

// Печать всех строк заказа (полиморфизм: virtual + override)
foreach (OrderLine line in order.GetLines())
{
    Console.WriteLine(line.GetProductDetails());
    Console.WriteLine($"Итого: {line.CalculateTotal()} руб.");
}

Console.WriteLine($"\nОбщая стоимость заказа: {order.CalculateTotal()} руб.");
Console.WriteLine($"Строк в заказе: {order.LineCount}");

// ============================= Быстрый тест =============================
// Сложное наследование FamilyPackLine.cs наследует StandardLine.cs:
//     FamilyPack -> StandardLine -> OrderLine (3 уровня иерархии)
// Множественное наследование FreeLine.cs реализует IBonus:
//     FreeLine наследует OrderLine И реализует интерфейс IBonus
Console.WriteLine($"\nБонус за акционные товары: {order.CalculateBonus()} баллов.");

// Базовый класс: строка заказа
public class OrderLine
{
    public int ProductId { get; set; }
    public string ProductName { get; set; }
    public decimal Price { get; set; }

    public OrderLine(int productId, string productName, decimal price)
    {
        ProductId = productId;
        ProductName = productName;
        Price = price;
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
    }

    public virtual string GetProductDetails()
    {
        return $"ID: {ProductId}, Название: {ProductName}, Цена: {Price} руб.";
    }
}

// Интерфейс: начисление бонусных баллов (для множественного наследования)
public interface IBonus
{
    decimal GetBonus();
}

// Производный класс: стандартная строка (с количеством единиц)
// ===================== ПРОСТОЕ наследование (1 уровень) =====================
public class StandardLine : OrderLine
{
    public int Units { get; set; }

    public StandardLine(int productId, string productName, decimal price, int units)
        : base(productId, productName, price)
    {
        Units = units;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    public override string GetProductDetails()
    {
        return $"{base.GetProductDetails()}, Кол-во: {Units} шт.";
    }
}

// Производный класс: семейная упаковка
// ===================== СЛОЖНОЕ наследование (2+ уровня) =====================
// FamilyPackLine -> StandardLine -> OrderLine
public class FamilyPackLine : StandardLine
{
    public int BonusUnits { get; set; }

    public FamilyPackLine(int productId, string productName, decimal price, int units, int bonusUnits)
        : base(productId, productName, price, units)
    {
        BonusUnits = bonusUnits;
    }

    // В семейной упаковке добавляются бесплатные единицы товара
    public override decimal CalculateTotal()
    {
        return Price * (Units + BonusUnits);
    }

    public override string GetProductDetails()
    {
        return $"{base.GetProductDetails()}, Бонус: +{BonusUnits} шт. бесплатно";
    }
}

// Производный класс: специальная строка (со скидкой)
// ===================== ПРОСТОЕ наследование (1 уровень) =====================
public class SpecialLine : OrderLine, IBonus
{
    public decimal Discount { get; set; }

    public SpecialLine(int productId, string productName, decimal price, decimal discount)
        : base(productId, productName, price)
    {
        Discount = discount;
    }

    // Применяем скидку к новой цене
    public override void UpdatePrice(decimal newPrice)
    {
        Price = newPrice * (1 - Discount / 100);
    }

    public override decimal CalculateTotal()
    {
        return Price * (1 - Discount / 100);
    }

    // Реализация интерфейса IBonus: 2% от стоимости со скидкой
    public decimal GetBonus()
    {
        return CalculateTotal() * 0.02m;
    }

    public override string GetProductDetails()
    {
        return $"{base.GetProductDetails()}, Скидка: {Discount}%";
    }
}

// Производный класс: бесплатная строка (с предоплатой)
// ===================== МНОЖЕСТВЕННОЕ наследование =====================
// FreeLine наследует OrderLine и реализует интерфейс IBonus (плюс как SpecialLine)
public class FreeLine : OrderLine, IBonus
{
    public decimal Prepayment { get; set; }

    public FreeLine(int productId, string productName, decimal price, decimal prepayment)
        : base(productId, productName, price)
    {
        Prepayment = prepayment;
    }

    public override decimal CalculateTotal()
    {
        return Price - Prepayment;
    }

    // Реализация интерфейса IBonus: 3% от стоимости за предоплату
    public decimal GetBonus()
    {
        return CalculateTotal() * 0.03m;
    }

    public override string GetProductDetails()
    {
        return $"{base.GetProductDetails()}, Предоплата: {Prepayment} руб.";
    }
}

// Класс заказа: хранит строки и взаимодействует с ними
// ===================== ВЗАИМОДЕЙСТВИЕ объектов =====================
// Order принимает строки через AddLine() и работает с ними через интерфейс IBonus
public class Order
{
    private List<OrderLine> _lines = new List<OrderLine>();

    public int LineCount
    {
        get { return _lines.Count; }
    }

    public void AddLine(OrderLine line)
    {
        _lines.Add(line);
    }

    public List<OrderLine> GetLines()
    {
        return _lines;
    }

    public decimal CalculateTotal()
    {
        decimal total = 0m;
        foreach (OrderLine line in _lines)
        {
            total += line.CalculateTotal();
        }
        return total;
    }

    // Взаимодействие через интерфейс: строка может и не являться IBonus
    public decimal CalculateBonus()
    {
        decimal bonus = 0m;
        foreach (OrderLine line in _lines)
        {
            if (line is IBonus)
            {
                bonus += ((IBonus)line).GetBonus();
            }
        }
        return bonus;
    }
}